# Recopilación del pipeline AMIA 2026: de YOLOv8s a RT-DETR-L + clasificador binario

Este cuaderno organiza la secuencia técnica que permitió pasar de una línea base con YOLOv8s a un sistema final basado en:

1. **Detector RT-DETR-L** entrenado sobre 14 clases de hallazgos.
2. **Ajuste de umbral de confianza** para controlar falsos positivos.
3. **Clasificador binario normal/anormal** para postprocesar el submission y convertir algunos falsos positivos a `No finding`.

El mejor resultado alcanzado fue:

| Etapa | Configuración | Public Score |
|---|---:|---:|
| YOLOv8s baseline | `conf=0.020`, `iou=0.40`, `max_det=300` | `0.29400` |
| RT-DETR-L | `conf=0.113`, `iou=0.40`, `max_det=300` | `0.49200` |
| RT-DETR-L + clasificador binario | `conf=0.113`, `p_abnormal < 0.001–0.003` → `No finding` | `0.49600` |

La idea central fue identificar que el problema principal ya no era solo detectar más cajas, sino **reducir falsos positivos en imágenes normales**.

## 0. Diccionario técnico de métricas usadas en el proyecto

Antes de entrar al pipeline, es importante entender qué miden las métricas que aparecieron durante el desarrollo. En este reto no se evalúa únicamente si una imagen está bien clasificada; se evalúa si el modelo **encuentra una lesión**, si la **ubica con una caja** y si asigna la **clase correcta**.

En términos físicos, cada radiografía se interpreta como una imagen donde una anomalía ocupa una región espacial. Por eso el modelo no solo entrega una etiqueta como *Cardiomegaly* o *Pleural effusion*, sino una caja rectangular sobre la zona donde cree que está el hallazgo.

El flujo completo tuvo tres niveles de métricas:

1. **Métricas de detección**: evalúan cajas, clases y confianza del detector RT-DETR o YOLO.
2. **Métricas de clasificación**: evalúan si una radiografía es normal o anormal.
3. **Métricas de submission/postprocesamiento**: evalúan cómo cambia el archivo final que se sube a Kaggle.


### 0.1 Métricas de localización: cajas e IoU

La detección de objetos usa cajas rectangulares para representar regiones de interés. En este proyecto, cada caja indica una posible anomalía en la radiografía.

| Métrica / variable | Qué significa | Qué representa físicamente en la radiografía | Cómo se usó en el proyecto |
|---|---|---|---|
| `x_min`, `y_min`, `x_max`, `y_max` | Coordenadas de una caja en la imagen original | Delimitan la zona donde aparece una posible lesión | Se usaron para construir las etiquetas de entrenamiento y para generar el submission final |
| `x_center`, `y_center`, `width`, `height` | Formato normalizado usado por Ultralytics | La misma caja, pero expresada como centro y tamaño relativo | Se usó para crear los archivos `.txt` de entrenamiento |
| `IoU` | Intersection over Union | Mide cuánto se solapa la caja predicha con la caja real del radiólogo | Si la caja cae en la zona correcta, el IoU sube; si está desplazada, baja |
| `iou_thres=0.40` | Umbral de IoU usado en inferencia/postproceso | Controla qué tan estricta es la comparación entre cajas solapadas | Se mantuvo fijo en `0.40` durante los mejores submissions |

La IoU se interpreta así:

$$
IoU = \frac{\text{área de intersección entre caja real y caja predicha}}{\text{área de unión entre caja real y caja predicha}}
$$

En términos físicos: si el modelo predice una caja encima del mismo hallazgo que marcó el radiólogo, la intersección es grande y el IoU será alto. Si la caja cae en otra región del tórax, la intersección será pequeña y el IoU será bajo.


### 0.2 Métricas de detección: precisión, recall y mAP

Durante el entrenamiento y validación de YOLOv8s y RT-DETR-L aparecieron métricas como `Box(P)`, `R`, `mAP50` y `mAP50-95`. Estas métricas son estándar en detección de objetos.

| Métrica | Qué significa | Qué hace físicamente | Interpretación en este reto |
|---|---|---|---|
| `Precision` o `Box(P)` | Proporción de predicciones correctas entre todas las predicciones hechas | Mide cuántas cajas predichas realmente corresponden a hallazgos | Si baja, el modelo está inventando demasiados hallazgos |
| `Recall` o `R` | Proporción de hallazgos reales que el modelo logró encontrar | Mide cuántas lesiones reales fueron detectadas | Si baja, el modelo está dejando pasar anomalías sin detectar |
| `mAP50` | Media de Average Precision usando IoU 0.50 | Resume precisión y recall considerando cajas correctas con IoU ≥ 0.50 | Sirve para comparar calidad general del detector en validación |
| `mAP50-95` | Promedio de mAP usando varios umbrales IoU entre 0.50 y 0.95 | Evalúa no solo encontrar la lesión, sino ubicarla cada vez con mayor precisión | Es más estricta que `mAP50` |
| `AP por clase` | Average Precision individual para cada patología | Mide qué tan bien detecta cada enfermedad por separado | Permitió ver clases fuertes como `Aortic enlargement` y clases difíciles como `Other lesion` |

Físicamente, estas métricas responden preguntas diferentes:

- **Precision**: de todas las zonas que el modelo marcó como lesión, ¿cuántas eran realmente lesiones?
- **Recall**: de todas las lesiones que existían, ¿cuántas encontró el modelo?
- **mAP**: ¿qué tan bueno es el modelo combinando confianza, clase y localización espacial?

En el reto, una alta precisión ayuda a reducir falsos positivos en imágenes normales. Un alto recall ayuda a no perder hallazgos reales. El balance entre ambas fue crítico para subir el score.


### 0.3 Métricas y parámetros de inferencia: `conf`, `max_det` y número de cajas

En los submissions, el parámetro más importante fue `conf_thres`. Este no cambia el modelo entrenado; solo cambia qué predicciones se aceptan al generar el CSV final.

| Parámetro / métrica | Qué significa | Qué hace físicamente | Efecto observado |
|---|---|---|---|
| `conf_thres` | Confianza mínima para aceptar una caja | Exige que el modelo esté más seguro antes de marcar una lesión | Al subirlo, aumentó `No finding` y mejoró el score hasta una meseta |
| `confidence` | Seguridad asignada por el detector a una caja | Qué tan convencido está el modelo de que esa región contiene una patología específica | Se guardó en el `PredictionString` junto con clase y caja |
| `max_det=300` | Máximo número de cajas permitidas por imagen | Evita que una radiografía tenga detecciones ilimitadas | Se mantuvo alto para no cortar hallazgos reales |
| `num_boxes` | Número de cajas predichas por imagen | Cantidad de regiones marcadas como posibles lesiones | Cuando era muy alto, indicaba exceso de falsos positivos |
| `mean_boxes` | Promedio de cajas por imagen | Nivel general de agresividad del detector | Bajó al aumentar `conf` y al usar el clasificador |

El comportamiento observado fue:

| Modelo / configuración | Resultado |
|---|---:|
| RT-DETR `conf=0.05` | 0.48500 |
| RT-DETR `conf=0.06` | 0.48700 |
| RT-DETR `conf=0.08` | 0.49000 |
| RT-DETR `conf=0.10` | 0.49100 |
| RT-DETR `conf=0.11` | 0.49200 |
| RT-DETR `conf=0.113` | 0.49200 |

Esto indicó que el detector estaba generando falsos positivos. Al subir `conf`, se eliminaron cajas débiles y el score mejoró. Sin embargo, después de `conf≈0.11`, el score entró en una meseta; por eso se agregó el clasificador binario normal/anormal.


### 0.4 Métrica de Kaggle: Public Score

El `Public Score` es la métrica final calculada por Kaggle sobre una parte del conjunto de test. No se puede calcular exactamente de forma local porque las etiquetas reales del test son privadas.

| Métrica | Qué significa | Qué mide físicamente | Cómo se usó |
|---|---|---|---|
| `Public Score` | Puntaje público del submission en Kaggle | Calidad final del sistema sobre imágenes ocultas | Fue la métrica principal para tomar decisiones |
| `PredictionString` | Cadena con todas las detecciones de una imagen | Lista de hallazgos detectados con clase, confianza y caja | Es el formato exigido por Kaggle |
| `14 1.0 0 0 1 1` | Predicción de `No finding` | Declara que no se detecta ninguna anomalía en la imagen | Se usó como fallback cuando no quedaban cajas |

En la vida real, este score representa qué tan confiable sería el sistema para apoyar la lectura de radiografías. Un score mayor implica mejor balance entre:

- detectar hallazgos reales;
- no inventar hallazgos en imágenes normales;
- localizar la región anatómica correcta;
- asignar la clase diagnóstica correcta.

La secuencia principal fue:

| Etapa | Mejor score |
|---|---:|
| YOLOv8s | 0.29400 |
| RT-DETR-L ajustando `conf` | 0.49200 |
| RT-DETR-L + clasificador binario | 0.49600 |


### 0.5 Métricas del clasificador binario normal/anormal

El clasificador binario fue entrenado para decidir si una radiografía completa era normal o anormal. A diferencia del detector, no devuelve cajas. Solo devuelve probabilidades globales de imagen.

| Métrica / variable | Qué significa | Qué hace físicamente | Uso en el pipeline |
|---|---|---|---|
| `top1_acc` | Accuracy de la clase más probable | Mide si el clasificador acertó normal/anormal en validación | El clasificador alcanzó `top1_acc = 0.946` |
| `top5_acc` | Accuracy si la clase correcta está entre las 5 más probables | En clasificación binaria siempre tiende a 1 porque solo hay 2 clases | No fue relevante para decidir el modelo |
| `p_normal` | Probabilidad de que la imagen sea normal | Qué tanto cree el clasificador que no hay hallazgos | Se usó como complemento de `p_abnormal` |
| `p_abnormal` | Probabilidad de que la imagen tenga alguna anomalía | Qué tanto cree el clasificador que existe al menos un hallazgo | Fue la variable clave del postprocesamiento |
| `abnormal_threshold` | Umbral aplicado a `p_abnormal` | Si el clasificador cree que la imagen es muy normal, se fuerza `No finding` | Subió el score de 0.49200 a 0.49600 |

El postprocesamiento aplicado fue:

```text
Si RT-DETR predice hallazgos
    y p_abnormal < threshold:
        convertir la imagen a No finding
```

Físicamente, esto significa que el detector puede marcar una región sospechosa, pero el clasificador global revisa toda la radiografía y actúa como filtro. Si la imagen completa parece normal, se eliminan esas cajas para reducir falsos positivos.

Los mejores resultados aparecieron con umbrales muy bajos:

| Postproceso | Score |
|---|---:|
| RT-DETR `conf=0.113` sin clasificador | 0.49200 |
| Clasificador `thr=0.0005` | 0.49500 |
| Clasificador `thr=0.001` | 0.49600 |
| Clasificador `thr=0.0015` | 0.49600 |
| Clasificador `thr=0.003` | 0.49600 |
| Clasificador `thr=0.007` | 0.49500 |

Esto muestra que el clasificador sí redujo falsos positivos, pero si el umbral se aumentaba demasiado empezaba a eliminar imágenes con hallazgos reales.


### 0.6 Métricas de pérdida durante entrenamiento

Durante el entrenamiento del detector aparecen pérdidas como `giou_loss`, `cls_loss` y `l1_loss`. Estas no son el score final, pero indican cómo aprende el modelo.

| Pérdida | Qué significa | Qué corrige físicamente |
|---|---|---|
| `giou_loss` | Error geométrico entre caja predicha y caja real | Ajusta la posición y tamaño de la caja para que cubra mejor el hallazgo |
| `cls_loss` | Error de clasificación | Ajusta la clase predicha para que coincida con la patología real |
| `l1_loss` | Distancia directa entre coordenadas de cajas | Corrige desplazamientos de bordes, centro, ancho y alto |
| `loss total` | Combinación de pérdidas | Resume qué tanto se equivoca el modelo durante entrenamiento |

En términos físicos:

- `giou_loss` y `l1_loss` corrigen **dónde** se dibuja la caja.
- `cls_loss` corrige **qué patología** se asigna a esa caja.

Una pérdida baja no garantiza mejor score público. Por eso se usó Kaggle como criterio final.


### 0.7 Métricas de diagnóstico del submission

Además del score de Kaggle, se analizaron métricas internas del CSV para entender el comportamiento del sistema.

| Métrica | Qué significa | Qué permitió decidir |
|---|---|---|
| `No finding` | Número de imágenes marcadas como normales | Sirvió para controlar falsos positivos |
| `% No finding` | Proporción de imágenes normales predichas | Se observó que el mejor rango estaba alrededor de 67–71% tras postprocesamiento |
| `Hallazgos` | Imágenes con al menos una caja predicha | Permitió medir agresividad del detector |
| `num_boxes` | Cajas por imagen | Mostró cuándo el modelo estaba generando demasiadas detecciones |
| `detecciones por clase` | Número de cajas predichas por cada anomalía | Ayudó a detectar clases sobrepredichas como `Other lesion`, `ILD` y `Pleural thickening` |
| `cajas inválidas` | Cajas con coordenadas incorrectas | Verificó que el submission no tuviera errores de formato |
| `cajas fuera de rango` | Cajas fuera de las dimensiones originales | Validó que la conversión de coordenadas fuera correcta |

Estas métricas no reemplazan el score de Kaggle, pero permiten explicar **por qué** una configuración sube o baja. Por ejemplo, cuando el porcentaje de `No finding` subía moderadamente, el score mejoraba porque se reducían falsos positivos; cuando subía demasiado, el score bajaba porque se perdían anomalías reales.


## 1. Instalación e importación de librerías

En esta primera etapa se prepara el entorno de Kaggle.  
Se instala `ultralytics`, se importan las librerías principales y se verifica si hay GPU disponible.

Esta celda se usa en los cuadernos de entrenamiento, inferencia y clasificación.

In [ ]:
# Instalamos Ultralytics, que contiene YOLOv8, RT-DETR y la API de entrenamiento/inferencia.
!pip install -q ultralytics

In [ ]:
# Importamos Path para manejar rutas de archivos de forma robusta.
from pathlib import Path

# Importamos pandas para leer y manipular archivos CSV.
import pandas as pd

# Importamos numpy para operaciones numéricas y recortes de coordenadas.
import numpy as np

# Importamos os para crear enlaces simbólicos y manejar rutas del sistema.
import os

# Importamos shutil para copiar archivos cuando no se pueda usar enlace simbólico.
import shutil

# Importamos gc para liberar memoria manualmente durante inferencia/entrenamiento.
import gc

# Importamos torch para verificar GPU y limpiar memoria CUDA.
import torch

# Importamos tqdm para mostrar barras de progreso.
from tqdm.auto import tqdm

# Importamos RTDETR para entrenar y usar el detector principal.
from ultralytics import RTDETR

# Importamos YOLO para usar modelos YOLO de clasificación.
from ultralytics import YOLO

# Mostramos si CUDA está disponible.
print("CUDA disponible:", torch.cuda.is_available())

# Si hay GPU disponible, mostramos cuántas hay y su nombre.
if torch.cuda.is_available():
    # Recorremos todas las GPUs detectadas.
    for i in range(torch.cuda.device_count()):
        # Imprimimos el índice y nombre de cada GPU.
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 2. Localización automática del dataset del reto

El dataset del reto fue agregado mediante **Add Data** en Kaggle.  
Como la ruta exacta puede cambiar, se busca automáticamente la carpeta que contiene:

- `train.csv`
- `sample_submission.csv`
- `img_size.csv`

Después se cargan los CSV principales y se indexan las imágenes `.png` por `image_id`.

In [ ]:
# Definimos la carpeta base donde Kaggle monta los datasets agregados con Add Data.
INPUT_DIR = Path("/kaggle/input")

# Creamos una lista vacía donde guardaremos carpetas candidatas del reto.
candidate_dirs = []

# Recorremos todos los archivos llamados train.csv dentro de /kaggle/input.
for train_csv in INPUT_DIR.rglob("train.csv"):
    # Tomamos la carpeta donde está cada train.csv encontrado.
    folder = train_csv.parent

    # Verificamos que en esa carpeta también estén sample_submission.csv e img_size.csv.
    if (folder / "sample_submission.csv").exists() and (folder / "img_size.csv").exists():
        # Si cumple, la guardamos como carpeta candidata del reto.
        candidate_dirs.append(folder)

# Si no se encontró ninguna carpeta candidata, detenemos la ejecución.
if len(candidate_dirs) == 0:
    # Lanzamos error claro para saber que falta agregar el dataset.
    raise FileNotFoundError("No se encontró la carpeta del reto en /kaggle/input")

# Seleccionamos la primera carpeta candidata encontrada.
DATA_DIR = candidate_dirs[0]

# Mostramos la carpeta seleccionada.
print("DATA_DIR:", DATA_DIR)

# Leemos el archivo de entrenamiento con anotaciones.
train_df = pd.read_csv(DATA_DIR / "train.csv")

# Leemos el archivo de test.
test_df = pd.read_csv(DATA_DIR / "test.csv")

# Leemos el archivo de dimensiones originales de cada imagen.
img_size_df = pd.read_csv(DATA_DIR / "img_size.csv")

# Leemos el formato de submission esperado por Kaggle.
sample_sub_df = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Mostramos las dimensiones de cada tabla.
print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("img_size_df:", img_size_df.shape)
print("sample_submission:", sample_sub_df.shape)

# Buscamos todas las imágenes PNG dentro del dataset.
png_files = list(DATA_DIR.rglob("*.png"))

# Creamos un diccionario que permite encontrar la ruta de una imagen usando su image_id.
image_path_dict = {p.stem: p for p in png_files}

# Mostramos cuántas imágenes PNG fueron encontradas.
print("PNG encontrados:", len(png_files))

## 3. Análisis de clases y decisión sobre `No finding`

En el reto hay 15 clases en `train.csv`, pero para detección se entrenan solo las clases `0` a `13`.  
La clase `14 = No finding` **no representa una caja real**, sino una imagen sin hallazgos.

Por eso se decidió:

- Entrenar el detector solo con clases `0–13`.
- Crear labels vacíos para imágenes `No finding`.
- En el submission, si el detector no predice cajas, escribir:

```text
14 1.0 0 0 1 1
```

In [ ]:
# Contamos cuántas imágenes únicas hay en train.
num_train_images = train_df["image_id"].nunique()

# Contamos cuántas imágenes únicas hay en test según sample_submission.
num_test_images = sample_sub_df["image_id"].nunique()

# Identificamos imágenes que tienen al menos una clase distinta de 14.
abnormal_image_ids = train_df.loc[train_df["class_id"] != 14, "image_id"].unique()

# Identificamos imágenes que aparecen como clase 14.
normal_image_ids = train_df.loc[train_df["class_id"] == 14, "image_id"].unique()

# Mostramos el total de imágenes de entrenamiento.
print("Imágenes únicas train:", num_train_images)

# Mostramos el total de imágenes de test.
print("Imágenes únicas test:", num_test_images)

# Mostramos cuántas imágenes tienen hallazgos.
print("Imágenes con hallazgos:", len(abnormal_image_ids))

# Mostramos cuántas imágenes tienen No finding.
print("Imágenes No finding:", len(normal_image_ids))

# Mostramos la distribución de filas por clase.
display(train_df["class_id"].value_counts().sort_index().to_frame("num_filas"))

## 4. Construcción del dataset en formato Ultralytics Detection

El formato requerido para detección en Ultralytics usa un archivo `.txt` por imagen.  
Cada línea del archivo de labels tiene:

```text
class_id x_center y_center width height
```

con coordenadas normalizadas entre `0` y `1`.

Un punto crítico del proyecto fue que las cajas de `train.csv` están en coordenadas originales, no necesariamente en 1024×1024. Por eso se normalizó usando:

- `dim0`: alto original.
- `dim1`: ancho original.

In [ ]:
# Definimos la carpeta donde se creará el dataset compatible con Ultralytics.
YOLO_DIR = Path("/kaggle/working/amia_yolo")

# Creamos las carpetas de imágenes y labels para train y val.
for split in ["train", "val"]:
    # Creamos carpeta de imágenes para el split.
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)

    # Creamos carpeta de labels para el split.
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Definimos los nombres de las 14 clases de detección.
class_names = [
    "Aortic enlargement",      # Clase 0.
    "Atelectasis",             # Clase 1.
    "Calcification",           # Clase 2.
    "Cardiomegaly",            # Clase 3.
    "Consolidation",           # Clase 4.
    "ILD",                     # Clase 5.
    "Infiltration",            # Clase 6.
    "Lung Opacity",            # Clase 7.
    "Nodule/Mass",             # Clase 8.
    "Other lesion",            # Clase 9.
    "Pleural effusion",        # Clase 10.
    "Pleural thickening",      # Clase 11.
    "Pneumothorax",            # Clase 12.
    "Pulmonary fibrosis",      # Clase 13.
]

# Creamos un diccionario con las dimensiones originales por image_id.
size_dict = img_size_df.set_index("image_id")[["dim0", "dim1"]].to_dict("index")

In [ ]:
# Importamos train_test_split para separar train y validación.
from sklearn.model_selection import train_test_split

# Creamos una tabla a nivel de imagen indicando si tiene hallazgo o no.
image_level = (
    train_df
    .groupby("image_id")["class_id"]
    .apply(lambda x: int((x != 14).any()))
    .reset_index()
)

# Renombramos las columnas de la tabla a nivel de imagen.
image_level.columns = ["image_id", "has_finding"]

# Separamos train y val manteniendo la proporción de imágenes normales/anormales.
train_ids, val_ids = train_test_split(
    image_level["image_id"].values,      # IDs de imagen a separar.
    test_size=0.2,                       # 20 % para validación.
    random_state=42,                     # Semilla fija para reproducibilidad.
    stratify=image_level["has_finding"]  # Estratificación normal/anormal.
)

# Convertimos los IDs de train a conjunto para búsqueda rápida.
train_ids = set(train_ids)

# Convertimos los IDs de val a conjunto para búsqueda rápida.
val_ids = set(val_ids)

# Mostramos el número de imágenes de entrenamiento.
print("Train images:", len(train_ids))

# Mostramos el número de imágenes de validación.
print("Val images:", len(val_ids))

In [ ]:
# Definimos una función para enlazar o copiar una imagen al dataset de trabajo.
def link_or_copy_image(img_id, split):
    # Obtenemos la ruta original de la imagen.
    src = image_path_dict[img_id]

    # Definimos la ruta destino dentro del dataset Ultralytics.
    dst = YOLO_DIR / "images" / split / src.name

    # Si el archivo ya existe, no hacemos nada.
    if dst.exists():
        return

    # Intentamos crear un enlace simbólico para ahorrar espacio.
    try:
        # Creamos el enlace simbólico.
        os.symlink(src, dst)

    # Si Kaggle no permite symlink en algún caso, copiamos la imagen.
    except:
        # Copiamos la imagen al destino.
        shutil.copy2(src, dst)


# Definimos una función para crear el archivo label .txt de una imagen.
def write_label(img_id, split):
    # Filtramos las filas de train_df correspondientes a la imagen.
    rows = train_df[train_df["image_id"] == img_id]

    # Definimos la ruta del archivo .txt de labels.
    label_path = YOLO_DIR / "labels" / split / f"{img_id}.txt"

    # Obtenemos el alto original de la imagen.
    dim0 = size_dict[img_id]["dim0"]

    # Obtenemos el ancho original de la imagen.
    dim1 = size_dict[img_id]["dim1"]

    # Creamos una lista vacía donde se guardarán las líneas YOLO.
    lines = []

    # Recorremos cada anotación de la imagen.
    for _, row in rows.iterrows():
        # Obtenemos la clase de la anotación.
        cls = int(row["class_id"])

        # Si la clase es No finding, no se escribe caja.
        if cls == 14:
            continue

        # Obtenemos la coordenada mínima en x.
        x_min = row["x_min"]

        # Obtenemos la coordenada mínima en y.
        y_min = row["y_min"]

        # Obtenemos la coordenada máxima en x.
        x_max = row["x_max"]

        # Obtenemos la coordenada máxima en y.
        y_max = row["y_max"]

        # Si alguna coordenada falta, saltamos esa anotación.
        if pd.isna(x_min) or pd.isna(y_min) or pd.isna(x_max) or pd.isna(y_max):
            continue

        # Calculamos el centro x normalizado por el ancho original.
        x_center = ((x_min + x_max) / 2) / dim1

        # Calculamos el centro y normalizado por el alto original.
        y_center = ((y_min + y_max) / 2) / dim0

        # Calculamos el ancho normalizado de la caja.
        width = (x_max - x_min) / dim1

        # Calculamos el alto normalizado de la caja.
        height = (y_max - y_min) / dim0

        # Limitamos x_center al rango válido [0, 1].
        x_center = np.clip(x_center, 0, 1)

        # Limitamos y_center al rango válido [0, 1].
        y_center = np.clip(y_center, 0, 1)

        # Limitamos width al rango válido [0, 1].
        width = np.clip(width, 0, 1)

        # Limitamos height al rango válido [0, 1].
        height = np.clip(height, 0, 1)

        # Si la caja no tiene área positiva, se ignora.
        if width <= 0 or height <= 0:
            continue

        # Agregamos la línea en formato YOLO.
        lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # Escribimos el archivo de labels; si lines está vacío, queda como archivo vacío.
    with open(label_path, "w") as f:
        # Escribimos todas las líneas separadas por salto de línea.
        f.write("\n".join(lines))

In [ ]:
# Recorremos todas las imágenes asignadas al conjunto de entrenamiento.
for img_id in train_ids:
    # Enlazamos o copiamos la imagen al split train.
    link_or_copy_image(img_id, "train")

    # Creamos el archivo de labels del split train.
    write_label(img_id, "train")

# Recorremos todas las imágenes asignadas al conjunto de validación.
for img_id in val_ids:
    # Enlazamos o copiamos la imagen al split val.
    link_or_copy_image(img_id, "val")

    # Creamos el archivo de labels del split val.
    write_label(img_id, "val")

# Definimos la ruta del archivo data.yaml.
yaml_path = YOLO_DIR / "data.yaml"

# Creamos el texto inicial del data.yaml.
yaml_text = f'''
path: {YOLO_DIR}
train: images/train
val: images/val
nc: 14
names:
'''

# Agregamos cada nombre de clase al data.yaml.
for name in class_names:
    # Agregamos una línea YAML con el nombre de clase.
    yaml_text += f"  - {name}\n"

# Escribimos el archivo data.yaml.
with open(yaml_path, "w") as f:
    # Guardamos el texto YAML en disco.
    f.write(yaml_text)

# Mostramos la ruta del YAML creado.
print("YAML creado:", yaml_path)

# Mostramos el contenido del YAML.
print(yaml_text)

## 5. Verificación del dataset de detección

Antes de entrenar, se verificó:

- Que cada imagen tenga su archivo `.txt`.
- Que las coordenadas estén entre `0` y `1`.
- Que las clases estén entre `0` y `13`.
- Que las imágenes `No finding` queden con archivo de labels vacío.

Esto permitió evitar entrenar con coordenadas mal escaladas.

In [ ]:
# Listamos imágenes de entrenamiento.
train_imgs = list((YOLO_DIR / "images" / "train").glob("*.png"))

# Listamos imágenes de validación.
val_imgs = list((YOLO_DIR / "images" / "val").glob("*.png"))

# Listamos labels de entrenamiento.
train_labels = list((YOLO_DIR / "labels" / "train").glob("*.txt"))

# Listamos labels de validación.
val_labels = list((YOLO_DIR / "labels" / "val").glob("*.txt"))

# Mostramos número de imágenes de train.
print("Train images:", len(train_imgs))

# Mostramos número de labels de train.
print("Train labels:", len(train_labels))

# Mostramos número de imágenes de val.
print("Val images:", len(val_imgs))

# Mostramos número de labels de val.
print("Val labels:", len(val_labels))

# Contamos cuántos labels vacíos hay en train.
empty_train = sum(1 for p in train_labels if p.read_text().strip() == "")

# Contamos cuántos labels vacíos hay en val.
empty_val = sum(1 for p in val_labels if p.read_text().strip() == "")

# Mostramos labels vacíos de train.
print("Train labels vacíos:", empty_train)

# Mostramos labels vacíos de val.
print("Val labels vacíos:", empty_val)

## 6. Línea base con YOLOv8s

Antes de usar RT-DETR-L se probó YOLOv8s como detector base.  
La mejor configuración de YOLOv8s fue:

| Modelo | Configuración | Public Score |
|---|---:|---:|
| YOLOv8s | `imgsz=640`, `conf=0.020`, `iou=0.40`, `max_det=300` | `0.29400` |

Este resultado sirvió como línea base, pero fue claramente inferior al RT-DETR-L.

In [ ]:
# Importamos YOLO para entrenar el modelo YOLOv8s.
from ultralytics import YOLO

# Liberamos memoria no usada de Python.
gc.collect()

# Liberamos memoria CUDA antes de entrenar.
torch.cuda.empty_cache()

# Definimos la ruta al archivo data.yaml de detección.
yaml_path = "/kaggle/working/amia_yolo/data.yaml"

# Cargamos YOLOv8s preentrenado.
model_yolo = YOLO("yolov8s.pt")

# Entrenamos YOLOv8s como línea base.
results_yolo = model_yolo.train(
    data=yaml_path,                             # Dataset Ultralytics.
    epochs=15,                                  # Número de épocas usado en la línea base.
    imgsz=640,                                  # Tamaño de entrada.
    batch=16,                                   # Batch usado para YOLOv8s.
    device=0,                                   # GPU 0.
    workers=2,                                  # Número de workers de carga.
    project="/kaggle/working/runs",             # Carpeta base de resultados.
    name="yolov8s_640_single_gpu",              # Nombre del experimento.
    pretrained=True,                            # Usar pesos preentrenados.
    patience=5,                                 # Early stopping.
    cache=False,                                # No cargar todo en caché.
    deterministic=False,                        # Permitir ejecución no determinista para velocidad.
    close_mosaic=0                              # No cerrar mosaic antes de terminar.
)

## 7. Entrenamiento del detector principal: RT-DETR-L

El salto fuerte de rendimiento vino al usar **RT-DETR-L**, entrenado sobre el mismo dataset de detección.

Configuración que produjo el modelo base:

| Parámetro | Valor |
|---|---:|
| Modelo inicial | `rtdetr-l.pt` |
| Épocas | `20` |
| Imagen | `640` |
| Batch | `8` |
| GPUs | `2` |
| `patience` | `8` |

Este modelo fue la base de los submissions que luego se optimizaron por umbral.

In [ ]:
# Liberamos memoria de Python antes de entrenar.
gc.collect()

# Liberamos memoria CUDA antes de cargar RT-DETR.
torch.cuda.empty_cache()

# Definimos la ruta del data.yaml.
yaml_path = "/kaggle/working/amia_yolo/data.yaml"

# Cargamos RT-DETR-L preentrenado.
model_rtdetr = RTDETR("rtdetr-l.pt")

# Entrenamos RT-DETR-L sobre el dataset del reto.
results_rtdetr = model_rtdetr.train(
    data=yaml_path,                                  # Ruta del dataset.
    epochs=20,                                      # Épocas totales del entrenamiento.
    imgsz=640,                                      # Tamaño de entrada.
    batch=8,                                        # Batch total, distribuido entre GPUs.
    device=[0, 1],                                  # Uso de dos GPUs.
    workers=4,                                      # Workers de carga de datos.
    project="/kaggle/working/runs",                 # Carpeta de salida.
    name="rtdetr_l_640_2gpu_e20_b8",                # Nombre del experimento.
    pretrained=True,                                # Usar pesos preentrenados.
    patience=8,                                     # Early stopping.
    cache=False,                                    # No usar caché completa.
    amp=True,                                       # Usar mixed precision.
    deterministic=False,                            # Permitir no determinismo para velocidad.
    save_period=5                                   # Guardar checkpoint cada 5 épocas.
)

# Definimos la ruta del mejor checkpoint.
best_rtdetr = Path("/kaggle/working/runs/rtdetr_l_640_2gpu_e20_b8/weights/best.pt")

# Definimos la ruta del último checkpoint.
last_rtdetr = Path("/kaggle/working/runs/rtdetr_l_640_2gpu_e20_b8/weights/last.pt")

# Mostramos la ruta del mejor checkpoint.
print("Best RT-DETR:", best_rtdetr)

# Verificamos que exista best.pt.
print("Existe best.pt:", best_rtdetr.exists())

# Verificamos que exista last.pt.
print("Existe last.pt:", last_rtdetr.exists())

## 8. Respaldo del modelo entrenado

Como Kaggle puede reiniciar la sesión, se copió el `best.pt` a `/kaggle/working` para descargarlo y reutilizarlo en otros cuadernos.

Este archivo se usó después como modelo original para inferencia:

```text
amia_rtdetr_l_640_2gpu_best.pt
```

In [ ]:
# Importamos shutil para copiar el archivo.
import shutil

# Importamos FileLink para crear un enlace de descarga dentro de Kaggle.
from IPython.display import FileLink

# Definimos la ruta del checkpoint best.pt original.
src = Path("/kaggle/working/runs/rtdetr_l_640_2gpu_e20_b8/weights/best.pt")

# Definimos la ruta destino con un nombre estable.
dst = Path("/kaggle/working/amia_rtdetr_l_640_2gpu_best.pt")

# Si el archivo fuente no existe, detenemos la ejecución.
if not src.exists():
    # Lanzamos error para no continuar sin modelo.
    raise FileNotFoundError("No existe best.pt del RT-DETR")

# Copiamos el modelo a la ruta destino.
shutil.copy(src, dst)

# Mostramos la ruta donde quedó respaldado.
print("Modelo respaldado en:", dst)

# Confirmamos que el archivo destino existe.
print("Existe:", dst.exists())

# Mostramos el tamaño aproximado del archivo.
print("Tamaño MB:", dst.stat().st_size / 1024 / 1024)

# Creamos enlace de descarga.
FileLink(str(dst))

## 9. Función de inferencia y generación de submission

El submission del reto tiene una columna `PredictionString`.

Cada caja se escribe como:

```text
class_id confidence xmin ymin xmax ymax
```

Si no hay detecciones, se escribe:

```text
14 1.0 0 0 1 1
```

La función siguiente convierte las cajas predichas por RT-DETR a las coordenadas originales del reto usando `dim0` y `dim1`.

In [ ]:
# Definimos una función general para generar submissions con RT-DETR.
def generar_submission_rtdetr(
    model,                  # Modelo RT-DETR cargado.
    output_path,            # Ruta donde se guardará el CSV final.
    conf_thres=0.113,       # Umbral de confianza del detector.
    iou_thres=0.40,         # Umbral IoU usado por la inferencia.
    max_det=300,            # Máximo número de detecciones por imagen.
    imgsz=640,              # Tamaño de entrada usado en inferencia.
    batch_size=1            # Batch de inferencia.
):
    # Creamos lista donde se guardará cada PredictionString.
    prediction_strings = []

    # Recorremos las imágenes de test por lotes.
    for start in tqdm(range(0, len(test_image_ids), batch_size)):
        # Calculamos el final del lote.
        end = min(start + batch_size, len(test_image_ids))

        # Extraemos los IDs del lote.
        batch_ids = test_image_ids[start:end]

        # Convertimos los IDs del lote a rutas de imagen.
        batch_paths = [str(image_path_dict[img_id]) for img_id in batch_ids]

        # Ejecutamos inferencia con RT-DETR.
        batch_results = model.predict(
            source=batch_paths,      # Imágenes del lote.
            imgsz=imgsz,            # Tamaño de inferencia.
            conf=conf_thres,        # Umbral de confianza.
            iou=iou_thres,          # Umbral IoU.
            max_det=max_det,        # Máximo de cajas.
            device=0,               # GPU 0 para inferencia.
            batch=batch_size,       # Tamaño de lote.
            verbose=False           # Evita imprimir por cada imagen.
        )

        # Recorremos resultados junto con sus image_id.
        for img_id, result in zip(batch_ids, batch_results):
            # Extraemos las cajas del resultado.
            boxes = result.boxes

            # Si no hay cajas, escribimos No finding.
            if boxes is None or len(boxes) == 0:
                # Agregamos PredictionString de No finding.
                prediction_strings.append("14 1.0 0 0 1 1")

                # Pasamos a la siguiente imagen.
                continue

            # Obtenemos el alto original de la imagen.
            dim0 = size_dict[img_id]["dim0"]

            # Obtenemos el ancho original de la imagen.
            dim1 = size_dict[img_id]["dim1"]

            # Obtenemos el alto y ancho que reporta Ultralytics en la imagen usada.
            img_h, img_w = result.orig_shape

            # Extraemos las cajas en formato xyxy.
            xyxy = boxes.xyxy.cpu().numpy()

            # Extraemos las confianzas.
            confs = boxes.conf.cpu().numpy()

            # Extraemos las clases.
            clss = boxes.cls.cpu().numpy().astype(int)

            # Creamos lista de partes del PredictionString.
            pred_parts = []

            # Recorremos clase, confianza y caja.
            for cls_id, conf, box in zip(clss, confs, xyxy):
                # Desempaquetamos coordenadas predichas.
                x1, y1, x2, y2 = box

                # Escalamos x1 a coordenadas originales.
                x1 = x1 * dim1 / img_w

                # Escalamos x2 a coordenadas originales.
                x2 = x2 * dim1 / img_w

                # Escalamos y1 a coordenadas originales.
                y1 = y1 * dim0 / img_h

                # Escalamos y2 a coordenadas originales.
                y2 = y2 * dim0 / img_h

                # Limitamos x1 al rango de la imagen original.
                x1 = float(np.clip(x1, 0, dim1))

                # Limitamos x2 al rango de la imagen original.
                x2 = float(np.clip(x2, 0, dim1))

                # Limitamos y1 al rango de la imagen original.
                y1 = float(np.clip(y1, 0, dim0))

                # Limitamos y2 al rango de la imagen original.
                y2 = float(np.clip(y2, 0, dim0))

                # Si la caja no tiene área positiva, la ignoramos.
                if x2 <= x1 or y2 <= y1:
                    # Saltamos caja inválida.
                    continue

                # Agregamos clase, confianza y coordenadas al PredictionString.
                pred_parts.extend([
                    str(int(cls_id)),        # Clase predicha.
                    f"{float(conf):.6f}",    # Confianza con 6 decimales.
                    f"{x1:.2f}",             # x mínimo.
                    f"{y1:.2f}",             # y mínimo.
                    f"{x2:.2f}",             # x máximo.
                    f"{y2:.2f}"              # y máximo.
                ])

            # Si después del filtrado no quedó ninguna caja, escribimos No finding.
            if len(pred_parts) == 0:
                # Agregamos No finding.
                prediction_strings.append("14 1.0 0 0 1 1")
            else:
                # Unimos todas las partes en un string.
                prediction_strings.append(" ".join(pred_parts))

        # Eliminamos resultados temporales para ahorrar memoria.
        del batch_results

        # Forzamos limpieza de memoria Python.
        gc.collect()

        # Limpiamos memoria CUDA.
        torch.cuda.empty_cache()

    # Copiamos el formato de sample_submission.
    submission_df = sample_sub_df.copy()

    # Reemplazamos la columna PredictionString con nuestras predicciones.
    submission_df["PredictionString"] = prediction_strings

    # Guardamos el CSV final.
    submission_df.to_csv(output_path, index=False)

    # Contamos cuántas imágenes fueron No finding.
    num_no_finding = (submission_df["PredictionString"] == "14 1.0 0 0 1 1").sum()

    # Mostramos ruta guardada.
    print("Guardado en:", output_path)

    # Mostramos forma del submission.
    print("Shape:", submission_df.shape)

    # Mostramos si hay nulos.
    print(submission_df.isna().sum())

    # Mostramos total de imágenes.
    print("Total imágenes:", len(submission_df))

    # Mostramos imágenes No finding.
    print("Predichas como No finding:", num_no_finding)

    # Mostramos imágenes con hallazgos.
    print("Predichas con hallazgos:", len(submission_df) - num_no_finding)

    # Mostramos porcentaje No finding.
    print("Porcentaje No finding:", num_no_finding / len(submission_df))

    # Visualizamos primeras filas.
    display(submission_df.head())

    # Retornamos el dataframe final.
    return submission_df

## 10. Optimización del umbral `conf` en RT-DETR-L

Se evaluaron varios umbrales de confianza.  
La tendencia mostró que subir `conf` reducía falsos positivos y mejoraba el score hasta una meseta.

| Detector | `conf` | No finding aprox. | Public Score |
|---|---:|---:|---:|
| RT-DETR-L | `0.05` | `56.62 %` | `0.48500` |
| RT-DETR-L | `0.06` | `59.55 %` | `0.48700` |
| RT-DETR-L | `0.08` | `63.36 %` | `0.49000` |
| RT-DETR-L | `0.10` | `65.80 %` | `0.49100` |
| RT-DETR-L | `0.113` | `67.79 %` | `0.49200` |

El mejor detector sin clasificador fue:

```text
RT-DETR-L conf=0.113, iou=0.40, max_det=300
```

In [ ]:
# Cargamos el modelo RT-DETR entrenado desde el checkpoint local.
model_rtdetr = RTDETR("/kaggle/working/runs/rtdetr_l_640_2gpu_e20_b8/weights/best.pt")

# Generamos el submission base que alcanzó 0.49200.
submission_rtdetr_0113 = generar_submission_rtdetr(
    model=model_rtdetr,                                                   # Modelo detector.
    output_path="/kaggle/working/submission_rtdetr_l_640_conf0113_iou04_max300.csv",  # Archivo de salida.
    conf_thres=0.113,                                                     # Mejor umbral detector.
    iou_thres=0.40,                                                       # IoU usado en inferencia.
    max_det=300,                                                          # Máximo de detecciones.
    imgsz=640,                                                            # Tamaño de imagen.
    batch_size=1                                                          # Batch seguro para memoria.
)

## 11. Intento de fine-tuning y descarte

Se probó continuar el entrenamiento desde el `best.pt`, pero el modelo fine-tuned bajó el score:

| Modelo | Configuración | Public Score |
|---|---:|---:|
| Original RT-DETR-L | `conf=0.113` | `0.49200` |
| Fine-tuned | `conf=0.11` | `0.47400` |
| Fine-tuned | `conf=0.18` | `0.47600` |

Se descartó porque el fine-tuning hizo el detector más agresivo y empeoró el equilibrio entre falsos positivos y verdaderos positivos.

## 12. Clasificador binario normal/anormal

El siguiente salto vino de entrenar un clasificador binario.  
La idea fue que RT-DETR detecta cajas, pero puede inventar hallazgos en imágenes normales.

El clasificador responde:

```text
¿La imagen es normal o anormal?
```

Luego se usa para corregir el submission:

```text
Si RT-DETR predice hallazgos, pero p_abnormal es muy baja,
entonces se convierte la imagen a No finding.
```

El clasificador YOLOv8s-cls obtuvo:

```text
top1_acc = 0.946 en validación
```

In [ ]:
# Creamos una tabla a nivel de imagen para clasificación binaria.
image_label_df = (
    train_df
    .groupby("image_id")["class_id"]                  # Agrupamos por imagen.
    .apply(lambda x: int((x != 14).any()))             # 1 si tiene alguna clase 0-13.
    .reset_index()                                     # Convertimos a dataframe.
)

# Renombramos columnas.
image_label_df.columns = ["image_id", "binary_label"]

# Convertimos 0/1 a nombres de carpeta.
image_label_df["class_name"] = image_label_df["binary_label"].map({
    0: "normal",                                       # Clase normal.
    1: "abnormal"                                      # Clase anormal.
})

# Mostramos la distribución de clases.
display(image_label_df["class_name"].value_counts().to_frame("num_images"))

In [ ]:
# Dividimos el dataset binario en train y validación.
train_cls_df, val_cls_df = train_test_split(
    image_label_df,                         # Dataframe con image_id y clase binaria.
    test_size=0.20,                         # 20 % validación.
    random_state=42,                        # Semilla fija.
    shuffle=True,                           # Mezclamos antes de separar.
    stratify=image_label_df["binary_label"] # Mantenemos proporción normal/anormal.
)

# Mostramos tamaño del split de entrenamiento.
print("Train:", train_cls_df.shape)

# Mostramos tamaño del split de validación.
print("Val:", val_cls_df.shape)

# Mostramos distribución en train.
display(train_cls_df["class_name"].value_counts().to_frame("train_images"))

# Mostramos distribución en val.
display(val_cls_df["class_name"].value_counts().to_frame("val_images"))

In [ ]:
# Definimos carpeta del dataset de clasificación.
CLS_DIR = Path("/kaggle/working/amia_binary_cls")

# Creamos carpetas train/val y normal/abnormal.
for split in ["train", "val"]:
    # Recorremos las dos clases.
    for cls_name in ["normal", "abnormal"]:
        # Creamos la carpeta correspondiente.
        (CLS_DIR / split / cls_name).mkdir(parents=True, exist_ok=True)


# Función para enlazar o copiar imágenes.
def safe_link_or_copy(src, dst):
    # Si el destino ya existe, no hacemos nada.
    if dst.exists():
        return

    # Intentamos crear enlace simbólico.
    try:
        # Creamos symlink.
        os.symlink(src, dst)

    # Si falla, copiamos el archivo.
    except:
        # Copiamos imagen al destino.
        shutil.copy(src, dst)


# Función para crear un split de clasificación.
def create_cls_split(df, split):
    # Recorremos filas del dataframe.
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Creando {split}"):
        # Obtenemos el image_id.
        img_id = row["image_id"]

        # Obtenemos nombre de clase: normal o abnormal.
        cls_name = row["class_name"]

        # Obtenemos ruta original de imagen.
        src_img = image_path_dict[img_id]

        # Definimos ruta destino.
        dst_img = CLS_DIR / split / cls_name / f"{img_id}.png"

        # Enlazamos o copiamos la imagen.
        safe_link_or_copy(src_img, dst_img)


# Creamos split de entrenamiento.
create_cls_split(train_cls_df, "train")

# Creamos split de validación.
create_cls_split(val_cls_df, "val")

# Mostramos ruta del dataset creado.
print("Dataset clasificación creado en:", CLS_DIR)

In [ ]:
# Liberamos memoria Python antes de entrenar el clasificador.
gc.collect()

# Liberamos memoria CUDA.
torch.cuda.empty_cache()

# Cargamos YOLOv8s de clasificación preentrenado.
model_cls = YOLO("yolov8s-cls.pt")

# Entrenamos el clasificador normal/anormal.
results_cls = model_cls.train(
    data=str(CLS_DIR),                                      # Dataset de clasificación.
    epochs=10,                                              # Épocas de entrenamiento.
    imgsz=640,                                              # Tamaño de entrada.
    batch=32,                                               # Batch de clasificación.
    device=0,                                               # GPU 0.
    workers=4,                                              # Workers de carga.
    project="/kaggle/working/runs_cls",                     # Carpeta de resultados.
    name="normal_abnormal_yolov8s_cls_640_e10",             # Nombre del experimento.
    pretrained=True,                                        # Usar pesos preentrenados.
    patience=3,                                             # Early stopping.
    cache=False,                                            # Sin caché completa.
    amp=True,                                               # Mixed precision.
    deterministic=False                                     # Permitir no determinismo.
)

# Ruta del mejor clasificador.
best_cls_path = Path("/kaggle/working/runs_cls/normal_abnormal_yolov8s_cls_640_e10/weights/best.pt")

# Ruta del último clasificador.
last_cls_path = Path("/kaggle/working/runs_cls/normal_abnormal_yolov8s_cls_640_e10/weights/last.pt")

# Mostramos ruta best.
print("Best classifier:", best_cls_path)

# Verificamos que exista.
print("Existe best:", best_cls_path.exists())

# Verificamos que exista last.
print("Existe last:", last_cls_path.exists())

## 13. Predicción de probabilidad `p_abnormal` en test

El clasificador se usa para cada imagen del test.  
Para cada `image_id` se guarda:

- `p_normal`
- `p_abnormal`

Después se cruza esta tabla con el submission base de RT-DETR.

In [ ]:
# Cargamos el mejor clasificador entrenado.
model_cls = YOLO(str(best_cls_path))

# Obtenemos el diccionario de nombres de clases del clasificador.
names = model_cls.names

# Inicializamos el índice de abnormal como None.
abnormal_idx = None

# Inicializamos el índice de normal como None.
normal_idx = None

# Recorremos nombres de clases para encontrar índices.
for idx, name in names.items():
    # Si el nombre es abnormal, guardamos su índice.
    if name.lower() == "abnormal":
        abnormal_idx = idx

    # Si el nombre es normal, guardamos su índice.
    if name.lower() == "normal":
        normal_idx = idx

# Mostramos índice de normal.
print("normal_idx:", normal_idx)

# Mostramos índice de abnormal.
print("abnormal_idx:", abnormal_idx)

# Si no encontramos abnormal, detenemos la ejecución.
if abnormal_idx is None:
    # Lanzamos error de configuración.
    raise ValueError("No encontré la clase abnormal en model_cls.names")

In [ ]:
# Obtenemos los IDs de test desde sample_submission.
test_image_ids = sample_sub_df["image_id"].tolist()

# Convertimos cada image_id a ruta de imagen.
test_paths = [str(image_path_dict[img_id]) for img_id in test_image_ids]

# Creamos lista donde guardaremos probabilidades.
cls_rows = []

# Definimos batch para inferencia del clasificador.
batch_size = 32

# Recorremos el test por lotes.
for start in tqdm(range(0, len(test_paths), batch_size)):
    # Calculamos final del lote.
    end = min(start + batch_size, len(test_paths))

    # Extraemos IDs del lote.
    batch_ids = test_image_ids[start:end]

    # Extraemos rutas del lote.
    batch_paths = test_paths[start:end]

    # Ejecutamos inferencia del clasificador.
    results = model_cls.predict(
        source=batch_paths,      # Imágenes del lote.
        imgsz=640,              # Tamaño de entrada.
        device=0,               # GPU.
        batch=batch_size,       # Batch.
        verbose=False           # Salida silenciosa.
    )

    # Recorremos resultados del lote.
    for img_id, r in zip(batch_ids, results):
        # Extraemos vector de probabilidades.
        probs = r.probs.data.cpu().numpy()

        # Obtenemos probabilidad abnormal.
        p_abnormal = float(probs[abnormal_idx])

        # Obtenemos probabilidad normal si existe su índice.
        p_normal = float(probs[normal_idx]) if normal_idx is not None else 1.0 - p_abnormal

        # Guardamos probabilidades de la imagen.
        cls_rows.append({
            "image_id": img_id,          # ID de imagen.
            "p_normal": p_normal,        # Probabilidad normal.
            "p_abnormal": p_abnormal     # Probabilidad abnormal.
        })

    # Eliminamos resultados temporales.
    del results

    # Liberamos memoria Python.
    gc.collect()

    # Liberamos memoria CUDA.
    torch.cuda.empty_cache()

# Convertimos la lista a dataframe.
cls_test_df = pd.DataFrame(cls_rows)

# Mostramos forma de la tabla.
print(cls_test_df.shape)

# Mostramos primeras filas.
display(cls_test_df.head())

# Mostramos distribución de probabilidad abnormal.
display(cls_test_df["p_abnormal"].describe())

## 14. Postprocesamiento con clasificador binario

Se toma el submission base de RT-DETR `conf=0.113`.

Regla:

```text
Si RT-DETR predijo hallazgos
y p_abnormal < threshold,
entonces convertir esa imagen a No finding.
```

Este postprocesamiento produjo el mejor score:

| Base | Threshold del clasificador | Public Score |
|---|---:|---:|
| RT-DETR `conf=0.113` | sin clasificador | `0.49200` |
| RT-DETR `conf=0.113` | `0.0005` | `0.49500` |
| RT-DETR `conf=0.113` | `0.001` | `0.49600` |
| RT-DETR `conf=0.113` | `0.0015` | `0.49600` |
| RT-DETR `conf=0.113` | `0.003` | `0.49600` |
| RT-DETR `conf=0.113` | `0.007` | `0.49500` |

El rango útil quedó entre `0.001` y `0.003`.

In [ ]:
# Definimos el string estándar para No finding.
NO_FINDING_STR = "14 1.0 0 0 1 1"

# Buscamos el submission base RT-DETR conf=0.113.
csv_candidates = list(Path("/kaggle/input").rglob("submission_rtdetr_l_640_conf0113_iou04_max300.csv"))

# También buscamos en /kaggle/working por si fue generado en el mismo cuaderno.
csv_candidates += list(Path("/kaggle/working").rglob("submission_rtdetr_l_640_conf0113_iou04_max300.csv"))

# Mostramos candidatos encontrados.
for p in csv_candidates:
    # Imprimimos ruta candidata.
    print(p)

# Si no hay candidatos, detenemos la ejecución.
if len(csv_candidates) == 0:
    # Lanzamos error indicando que falta agregar el CSV base.
    raise FileNotFoundError("No se encontró el submission base conf=0.113")

# Tomamos el primer candidato.
base_sub_path = csv_candidates[0]

# Leemos el CSV base.
base_sub = pd.read_csv(base_sub_path)

# Mostramos forma del submission base.
print("base_sub:", base_sub.shape)

# Mostramos primeras filas.
display(base_sub.head())

In [ ]:
# Contamos No finding en el submission base.
base_no_finding = (base_sub["PredictionString"] == NO_FINDING_STR).sum()

# Mostramos cantidad No finding base.
print("No finding base:", base_no_finding)

# Mostramos cantidad de imágenes con hallazgos base.
print("Hallazgos base:", len(base_sub) - base_no_finding)

# Mostramos porcentaje No finding base.
print("Porcentaje No finding base:", base_no_finding / len(base_sub))

# Cruzamos el submission base con las probabilidades del clasificador.
df_check = base_sub.merge(cls_test_df, on="image_id", how="left")

# Si alguna imagen no tiene probabilidad, detenemos ejecución.
if df_check["p_abnormal"].isna().sum() > 0:
    # Error si falta alguna predicción del clasificador.
    raise ValueError("Hay imágenes sin probabilidad del clasificador")

# Creamos indicador de No finding según RT-DETR.
df_check["rtdetr_no_finding"] = df_check["PredictionString"] == NO_FINDING_STR

# Creamos indicador de hallazgo según RT-DETR.
df_check["rtdetr_has_finding"] = ~df_check["rtdetr_no_finding"]

# Definimos umbrales que se evaluaron.
thresholds = [0.0005, 0.001, 0.0015, 0.003, 0.007]

# Creamos lista para guardar resumen.
rows = []

# Recorremos umbrales.
for thr in thresholds:
    # Identificamos imágenes con hallazgos RT-DETR pero baja p_abnormal.
    mask_change = (
        df_check["rtdetr_has_finding"] &
        (df_check["p_abnormal"] < thr)
    )

    # Contamos cuántas imágenes serían convertidas a No finding.
    changed = mask_change.sum()

    # Calculamos No finding final.
    final_no_finding = base_no_finding + changed

    # Guardamos resumen del umbral.
    rows.append({
        "abnormal_threshold": thr,                            # Umbral evaluado.
        "imagenes_cambiadas_a_no_finding": changed,           # Imágenes modificadas.
        "no_finding_final": final_no_finding,                 # Total No finding final.
        "porcentaje_no_finding_final": final_no_finding / len(df_check), # Porcentaje final.
        "hallazgos_final": len(df_check) - final_no_finding   # Imágenes restantes con hallazgos.
    })

# Convertimos el resumen a dataframe.
threshold_table = pd.DataFrame(rows)

# Mostramos tabla de umbrales.
display(threshold_table)

In [ ]:
# Definimos función para aplicar el clasificador al submission base.
def postprocess_with_binary_classifier(
    base_sub,              # Submission base de RT-DETR.
    cls_test_df,           # Probabilidades normal/anormal del test.
    output_path,           # Ruta del CSV postprocesado.
    abnormal_threshold     # Umbral de p_abnormal.
):
    # Cruzamos submission y probabilidades por image_id.
    df = base_sub.merge(cls_test_df, on="image_id", how="left")

    # Si faltan probabilidades, detenemos la ejecución.
    if df["p_abnormal"].isna().sum() > 0:
        # Error para evitar CSV incompleto.
        raise ValueError("Hay imágenes sin probabilidad del clasificador")

    # Contamos No finding original.
    original_no_finding = (df["PredictionString"] == NO_FINDING_STR).sum()

    # Marcamos imágenes con hallazgo RT-DETR pero baja probabilidad abnormal.
    mask_force_normal = (
        (df["PredictionString"] != NO_FINDING_STR) &
        (df["p_abnormal"] < abnormal_threshold)
    )

    # Contamos cuántas imágenes se modificarán.
    changed = mask_force_normal.sum()

    # Convertimos esas imágenes a No finding.
    df.loc[mask_force_normal, "PredictionString"] = NO_FINDING_STR

    # Contamos No finding final.
    final_no_finding = (df["PredictionString"] == NO_FINDING_STR).sum()

    # Dejamos solo columnas requeridas por Kaggle.
    out_df = df[["image_id", "PredictionString"]].copy()

    # Guardamos CSV final.
    out_df.to_csv(output_path, index=False)

    # Mostramos ruta guardada.
    print("Guardado:", output_path)

    # Mostramos umbral usado.
    print("Threshold abnormal:", abnormal_threshold)

    # Mostramos cuántas imágenes cambiaron.
    print("Imágenes cambiadas a No finding:", changed)

    # Mostramos No finding original.
    print("No finding original:", original_no_finding)

    # Mostramos No finding final.
    print("No finding final:", final_no_finding)

    # Mostramos porcentaje No finding final.
    print("Porcentaje No finding final:", final_no_finding / len(out_df))

    # Mostramos cantidad final de hallazgos.
    print("Hallazgos final:", len(out_df) - final_no_finding)

    # Mostramos forma del dataframe.
    print("Shape:", out_df.shape)

    # Verificamos nulos.
    print(out_df.isna().sum())

    # Mostramos primeras filas.
    display(out_df.head())

    # Retornamos el submission final.
    return out_df

In [ ]:
# Generamos submission final usando threshold 0.001.
sub_cls_001 = postprocess_with_binary_classifier(
    base_sub=base_sub,                                             # Submission RT-DETR conf=0.113.
    cls_test_df=cls_test_df,                                       # Probabilidades del clasificador.
    output_path="/kaggle/working/submission_rtdetr_conf0113_cls_thr001.csv", # CSV final.
    abnormal_threshold=0.001                                       # Umbral exitoso.
)

# Generamos submission final usando threshold 0.003.
sub_cls_003 = postprocess_with_binary_classifier(
    base_sub=base_sub,                                             # Submission RT-DETR conf=0.113.
    cls_test_df=cls_test_df,                                       # Probabilidades del clasificador.
    output_path="/kaggle/working/submission_rtdetr_conf0113_cls_thr003.csv", # CSV final.
    abnormal_threshold=0.003                                       # Umbral exitoso.
)

## 15. Resultado final y lectura técnica

La mejora no vino de seguir aumentando el `conf`, sino de agregar una segunda etapa de decisión:

1. **RT-DETR-L** localiza hallazgos.
2. **Clasificador binario** decide si la imagen completa parece normal o anormal.
3. Si el detector predice hallazgos, pero el clasificador está muy seguro de que es normal, se fuerza `No finding`.

El mejor resultado actual:

```text
RT-DETR-L original
conf = 0.113
iou = 0.40
max_det = 300
+
YOLOv8s-cls normal/abnormal
p_abnormal threshold entre 0.001 y 0.003

Public Score = 0.49600
```

Interpretación:

- El detector solo llegó a `0.49200`.
- El clasificador binario corrigió falsos positivos.
- La mejora fue de `+0.00400`.
- El siguiente paso recomendado sería un clasificador **multi-label** de 14 clases para filtrar cajas por clase.

## 16. Resultados consolidados

| Etapa | Configuración | Decisión | Public Score |
|---|---:|---|---:|
| YOLOv8s | `conf=0.020` | Línea base descartada | `0.29400` |
| RT-DETR-L | `conf=0.05` | Muy agresivo | `0.48500` |
| RT-DETR-L | `conf=0.08` | Mejoró al limpiar falsos positivos | `0.49000` |
| RT-DETR-L | `conf=0.10` | Buen equilibrio | `0.49100` |
| RT-DETR-L | `conf=0.113` | Mejor detector solo | `0.49200` |
| Fine-tuned RT-DETR-L | `conf=0.11–0.18` | Descartado | `0.474–0.476` |
| RT-DETR-L + clasificador | `thr=0.0005` | Conservador | `0.49500` |
| RT-DETR-L + clasificador | `thr=0.001–0.003` | Mejor sistema | `0.49600` |
| RT-DETR-L + clasificador | `thr=0.007` | Demasiado agresivo | `0.49500` |

Conclusión :

> El mayor salto no se consiguió ajustando más el detector, sino incorporando una etapa secundaria de clasificación normal/anormal para reducir falsos positivos en imágenes que el detector marcaba como patológicas.